# Linear Regression — House Price Prediction
### Abrar Jawad | June 2026

---

## What This Notebook Is

This is the ML phase of my house price analysis. Project 02 was pure EDA —
exploring the data, finding correlations, understanding distributions. Now I'm
using those findings to actually build a model that predicts house prices.

I'm using the same Kaggle dataset: House Prices: Advanced Regression Techniques
(1,460 houses, 81 features). The target variable is SalePrice.

This is my first ever ML model.

## Why Linear Regression First

Linear regression is the foundation of supervised ML. Before trees, forests,
or neural networks, you need to understand the simplest case — fitting a line
through data to make predictions. Everything else builds on top of this.

## What I Already Know Going In

From Project 02 EDA, I already know which features correlate most with SalePrice:
- OverallQual (r = 0.791) — strongest predictor
- TotalSF (r = 0.779) — engineered feature
- GrLivArea (r = 0.709)
- GarageCars (r = 0.640)
- GarageArea (r = 0.623)

So feature selection isn't guesswork here — it's backed by prior analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics

In [ ]:
df = pd.read_csv("../data/raw/train.csv")
print(df.shape)
df.head()

## Selecting Features and Target


In [ ]:
features = ['GrLivArea', 'OverallQual', 'TotalBsmtSF', 'GarageCars', 'YearBuilt']
df_model = df[features + ['SalePrice']].dropna()

x = df_model[features]
y = df_model['SalePrice']

print(x.shape)
print(y.shape)

## Train_Test Split


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print(x_train.shape)
print(x_test.shape)

## Training the Model


In [ ]:
model = LinearRegression()
model.fit(x_train, y_train)

print(f"Intercept: {model.intercept_}")
print(f"Coefficients: {model.coef_}")

## Making Predictions


In [ ]:
y_pred = model.predict(x_test)

print(y_pred[:5])
print(y_test.values[:5])

## Evaluating Performance

In [ ]:
mae = metrics.mean_absolute_error(y_test, y_pred)
mse = metrics.mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = metrics.r2_score(y_test, y_pred)

print(f"MAE: {mae:.0f}")
print(f"RMSE: {rmse:.0f}")
print(f"R²: {r2:.4f}")

## Actual vs Predicted Plot

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.4, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title("Actual vs Predicted Sale Price")
plt.tight_layout()
plt.savefig('../visuals/lr_actual_vs_predicted.png', dpi=150)
plt.show()

## Linear Regression — What I Found

### Model & Features
I used 5 features: GrLivArea, OverallQual, TotalBsmtSF, GarageCars, and YearBuilt.
These were chosen based on the correlation heatmap from Project 02 EDA — I already
knew these were the strongest predictors before building the model.

### Coefficients — What the Model Learned
The model assigned the highest coefficient to OverallQual (~$20,392 per quality point).
This matched exactly what EDA showed — OverallQual had the highest correlation with
SalePrice (r = 0.791). The EDA predicted this before I even touched scikit-learn.

Other coefficients made real-world sense too:
- Each extra sq ft of living area adds ~$49
- Each extra garage car space adds ~$15k
- Each newer year built adds ~$316

### Evaluation Numbers
- MAE: $25,415 — on average, my predictions are off by ~$25k per house
- RMSE: $39,763 — noticeably higher than MAE, which means a few houses have
  very large errors dragging this up
- R²: 0.794 — the model explains 79.4% of the variation in house prices
  using just 5 features. The remaining 20.6% is things the model couldn't capture.

### What the Scatter Plot Showed
Dots clustered well around the perfect prediction line up to ~$300k. Beyond that,
the model consistently underpredicted — almost all expensive houses were below
the red line. The worst case was a house that actually sold for $750k but the
model predicted $460k — a $290k error on a single house. There was also one
house predicted at nearly $0, likely an outlier with unusual feature values.

The gap between RMSE and MAE is explained by exactly these extreme cases.

### Main Limitation
The model works well in the middle but struggles at the extremes. This is a known
limitation of linear regression on skewed price data. The fix is to train on
log(SalePrice) instead of raw SalePrice — the same log transform I applied in
Project 02. That's the next improvement to try.

## Improvement — Log Transformation of Target Variable

The baseline model struggled with expensive houses — consistently underpredicting
them. This is because SalePrice is right-skewed. Training on log(SalePrice)
instead of raw SalePrice compresses the extreme values and makes the target
more symmetric. After predictions, we reverse the log to get back to dollars.